In [2]:
from pathlib import Path
import pandas as pd
import csv

# ============================================================
# KLASÖR
# ============================================================

DATA_DIR = Path(
    r"D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\reuters_annotation_batch"
)

# ============================================================
# BEKLENEN KOLONLAR
# ============================================================

DATA_COLUMNS = [
    "annotation_id",
    "date",
    "finbert_score",
    "finbert_label",
    "finbert_confidence",
    "TEXT",
    "gemini_label"
]

rows = []
errors = []


# ============================================================
# TXT OKUMA
# ============================================================

def read_txt(file):

    result = []

    with open(
        file,
        "r",
        encoding="utf-8-sig",
        errors="replace"
    ) as f:

        for line_number, line in enumerate(f, start=1):

            line = line.strip()

            if not line:
                continue

            # Header
            if line.lower().startswith("annotation_id"):
                continue

            try:

                parts = next(csv.reader([line]))

                # Normal
                if len(parts) == 7:

                    annotation_id = parts[0].strip()
                    date = parts[1].strip()
                    finbert_score = parts[2].strip()
                    finbert_label = parts[3].strip()
                    finbert_confidence = parts[4].strip()
                    text = parts[5].strip()
                    gemini_label = parts[6].strip()

                # TEXT içerisinde virgül varsa
                elif len(parts) > 7:

                    annotation_id = parts[0].strip()
                    date = parts[1].strip()
                    finbert_score = parts[2].strip()
                    finbert_label = parts[3].strip()
                    finbert_confidence = parts[4].strip()

                    text = ",".join(
                        x.strip()
                        for x in parts[5:-1]
                    )

                    gemini_label = parts[-1].strip()

                else:

                    errors.append({
                        "file": file.name,
                        "line": line_number,
                        "reason": f"{len(parts)} kolon bulundu",
                        "content": line
                    })

                    continue

                result.append({
                    "annotation_id": annotation_id,
                    "date": date,
                    "finbert_score": finbert_score,
                    "finbert_label": finbert_label,
                    "finbert_confidence": finbert_confidence,
                    "TEXT": text,
                    "gemini_label": gemini_label,
                    "source_file": file.name
                })

            except Exception as e:

                errors.append({
                    "file": file.name,
                    "line": line_number,
                    "reason": str(e),
                    "content": line
                })

    return result


# ============================================================
# DOSYALARI BUL
# ============================================================

txt_files = sorted(DATA_DIR.glob("*.txt"))
csv_files = sorted(DATA_DIR.glob("*.csv"))

# Prompt'u çıkar
txt_files = [
    f for f in txt_files
    if f.name.lower() != "000_prompt.txt"
]

print("=" * 80)
print("DOSYA SAYILARI")
print("=" * 80)

print("TXT :", len(txt_files))
print("CSV :", len(csv_files))
print("TOPLAM:", len(txt_files) + len(csv_files))


# ============================================================
# TXT DOSYALARI
# ============================================================

print("\n" + "=" * 80)
print("TXT DOSYALARI")
print("=" * 80)

for file in txt_files:

    try:

        data = read_txt(file)

        rows.extend(data)

        print(
            f"{file.name:15} -> {len(data):5} kayıt"
        )

    except Exception as e:

        print(
            f"{file.name:15} -> HATA: {e}"
        )

        errors.append({
            "file": file.name,
            "line": None,
            "reason": str(e),
            "content": None
        })


# ============================================================
# CSV DOSYALARI
# ============================================================

print("\n" + "=" * 80)
print("CSV DOSYALARI")
print("=" * 80)

for file in csv_files:

    try:

        df = pd.read_csv(
            file,
            dtype=str,
            encoding="utf-8-sig",
            on_bad_lines="warn"
        )

        # Kolon isimlerini temizle
        df.columns = (
            df.columns
            .astype(str)
            .str.strip()
            .str.replace("\ufeff", "", regex=False)
        )

        print(
            f"{file.name:55} -> {len(df):5} kayıt"
        )

        # Beklenen kolonların mevcut olanlarını kullan
        for _, row in df.iterrows():

            rows.append({
                "annotation_id": row.get(
                    "annotation_id",
                    pd.NA
                ),

                "date": row.get(
                    "date",
                    pd.NA
                ),

                "finbert_score": row.get(
                    "finbert_score",
                    pd.NA
                ),

                "finbert_label": row.get(
                    "finbert_label",
                    pd.NA
                ),

                "finbert_confidence": row.get(
                    "finbert_confidence",
                    pd.NA
                ),

                "TEXT": row.get(
                    "TEXT",
                    pd.NA
                ),

                "gemini_label": row.get(
                    "gemini_label",
                    pd.NA
                ),

                "source_file": file.name
            })

    except Exception as e:

        print(
            f"{file.name:55} -> HATA: {e}"
        )

        errors.append({
            "file": file.name,
            "line": None,
            "reason": str(e),
            "content": None
        })


# ============================================================
# FINAL DATAFRAME
# ============================================================

final_df = pd.DataFrame(rows)


# ============================================================
# TEMİZLİK
# ============================================================

final_df = final_df.replace(
    [
        "nan",
        "NaN",
        "None",
        "",
        " "
    ],
    pd.NA
)


# ============================================================
# VERİ TİPLERİ
# ============================================================

final_df["date"] = pd.to_datetime(
    final_df["date"],
    errors="coerce"
)

final_df["finbert_score"] = pd.to_numeric(
    final_df["finbert_score"],
    errors="coerce"
)

final_df["finbert_confidence"] = pd.to_numeric(
    final_df["finbert_confidence"],
    errors="coerce"
)


# ============================================================
# STRING KOLONLARI
# ============================================================

for col in [
    "annotation_id",
    "finbert_label",
    "TEXT",
    "gemini_label",
    "source_file"
]:

    final_df[col] = (
        final_df[col]
        .astype("string")
        .str.strip()
    )


# ============================================================
# SIRALAMA
# ============================================================

final_df = final_df.sort_values(
    "annotation_id",
    na_position="last"
).reset_index(drop=True)


# ============================================================
# SONUÇ
# ============================================================

print("\n" + "=" * 80)
print("FINAL DATAFRAME")
print("=" * 80)

print("Toplam kayıt :", len(final_df))
print("Toplam kolon :", len(final_df.columns))
print(
    "Benzersiz annotation_id :",
    final_df["annotation_id"].nunique()
)

print("\nKolonlar:")
print(final_df.columns.tolist())


# ============================================================
# DOSYA TÜRÜNE GÖRE KAYIT
# ============================================================

print("\n" + "=" * 80)
print("KAYNAK DOSYA İSTATİSTİĞİ")
print("=" * 80)

print(
    final_df["source_file"]
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# DUPLICATE
# ============================================================

duplicates = final_df[
    final_df["annotation_id"].duplicated(
        keep=False
    )
]

print("\n" + "=" * 80)
print("DUPLICATE")
print("=" * 80)

print(
    "Duplicate satır sayısı:",
    len(duplicates)
)

print(
    "Duplicate annotation sayısı:",
    duplicates["annotation_id"].nunique()
)


# ============================================================
# HATALAR
# ============================================================

errors_df = pd.DataFrame(errors)

print("\n" + "=" * 80)
print("HATALAR")
print("=" * 80)

print(
    "Hatalı kayıt/dosya:",
    len(errors_df)
)


# ============================================================
# İLK 10
# ============================================================

print("\n" + "=" * 80)
print("İLK 10 KAYIT")
print("=" * 80)

print(
    final_df.head(10).to_string(index=False)
)

final_df = final_df.drop_duplicates(
    subset="annotation_id",
    keep="first"
).reset_index(drop=True)

print("Kalan kayıt:", len(final_df))
print("Benzersiz annotation_id:", final_df["annotation_id"].nunique())

DOSYA SAYILARI
TXT : 30
CSV : 63
TOPLAM: 93

TXT DOSYALARI
001.txt         ->    50 kayıt
002.txt         ->    50 kayıt
003.txt         ->    50 kayıt
004.txt         ->    50 kayıt
005.txt         ->    50 kayıt
006.txt         ->    50 kayıt
007.txt         ->    50 kayıt
008.txt         ->    50 kayıt
009.txt         ->    50 kayıt
010.txt         ->    50 kayıt
011.txt         ->    50 kayıt
012.txt         ->    50 kayıt
013.txt         ->    50 kayıt
014.txt         ->    50 kayıt
015.txt         ->    50 kayıt
016.txt         ->    50 kayıt
017.txt         ->    50 kayıt
018.txt         ->    50 kayıt
019.txt         ->    50 kayıt
020.txt         ->    50 kayıt
021.txt         ->    50 kayıt
022.txt         ->    50 kayıt
023.txt         ->    50 kayıt
024.txt         ->    50 kayıt
025.txt         ->    50 kayıt
026.txt         ->    50 kayıt
027.txt         ->    50 kayıt
028.txt         ->    50 kayıt
029.txt         ->     0 kayıt
030.txt         ->     0 kayıt

CSV DOSYAL

In [3]:
final_df

,annotation_id,date,finbert_score,finbert_label,finbert_confidence,TEXT,gemini_label,source_file
0,REUTERS_ANN_00001,2007-04-12,-1.0,negative,0.683211,s.africa watchdog to probe gold fields bid report,neutral,001.txt
1,REUTERS_ANN_00002,2007-04-26,0.0,neutral,0.847032,new barbie girls sashay into view with mp-3,neutral,001.txt
2,REUTERS_ANN_00003,2006-12-17,0.0,neutral,0.865116,"stocks await data,mergers and santa",neutral,001.txt
3,REUTERS_ANN_00004,2007-01-03,-1.0,negative,0.960498,canadian dollar falls to 9-mth lows vs u.s. do...,negative,001.txt
4,REUTERS_ANN_00005,2007-02-05,-1.0,negative,0.964671,"techs buckle under microsoft,wal-mart helps dow",neutral,001.txt
...,...,...,...,...,...,...,...,...
5035,SP500_ANN_1436,2013-05-14,NaN,neutral,0.678842,Apple’s App Store hits 50 billion downloads,negative,sp500_batch_144_gemini_completed.csv
5036,SP500_ANN_1437,2011-08-30,NaN,positive,0.899694,DreamWorks Strikes 'Kung Fu Panda' Distributio...,positive,sp500_batch_144_gemini_completed.csv
5037,SP500_ANN_1438,2024-01-09,NaN,positive,0.938962,Gold has outperformed the S&P 500 in the 21st ...,positive,sp500_batch_144_gemini_completed.csv
5038,SP500_ANN_1439,2023-02-23,NaN,negative,0.702293,"S&P 500 Continues Losing Streak, NASDAQ Bucks ...",negative,sp500_batch_144_gemini_completed.csv


In [4]:
expected_ids = {
    f"REUTERS_ANN_{i:05d}"
    for i in range(1, 5001)
}

actual_ids = set(
    final_df["annotation_id"].dropna()
)

missing_ids = sorted(
    expected_ids - actual_ids
)

print("Eksik annotation sayısı:", len(missing_ids))

print("\nEksik annotation'lar:")
for x in missing_ids:
    print(x)

Eksik annotation sayısı: 0

Eksik annotation'lar:


In [5]:
import pandas as pd

reuters_path = r"D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\REUTERS_annotation_all_clean.csv"

reuters_df = pd.read_csv(
    reuters_path,
    encoding="utf-8-sig"
)

print("Satır sayısı:", len(reuters_df))
print("Kolonlar:")
print(reuters_df.columns.tolist())

print("\nİlk 10 kayıt:")
display(reuters_df.head(10))

Satır sayısı: 5000
Kolonlar:
['annotation_id', 'date', 'text_en', 'text_tr', 'finbert_score', 'finbert_label', 'finbert_confidence', 'chatgpt_label', 'chatgpt_score', 'chatgpt_confidence', 'chatgpt_reason_tr', 'final_label', 'finbert_correctness', 'review_needed', 'row_note', 'batch_no', 'n_words', 'text_len']

İlk 10 kayıt:


,annotation_id,date,text_en,text_tr,finbert_score,finbert_label,finbert_confidence,chatgpt_label,chatgpt_score,chatgpt_confidence,chatgpt_reason_tr,final_label,finbert_correctness,review_needed,row_note,batch_no,n_words,text_len
0,REUTERS_ANN_00001,2007-04-12,s.africa watchdog to probe gold fields bid report,Güney Afrika rekabet kurumu Gold Fields teklif...,-1,negative,0.683211,negative,-1.0,0.86,Bir satın alma teklifinin rekabet kurumu taraf...,negative,same,no,FinBERT etiketi bağlamla uyumlu; 'watchdog to ...,1,8,49
1,REUTERS_ANN_00002,2007-04-26,new barbie girls sashay into view with mp-3,Yeni Barbie Girls MP3 ile sahneye çıkıyor,0,neutral,0.847032,positive,1.0,0.76,Yeni MP3 özellikli ürün lansmanı ürün yeniliği...,positive,different,yes,FinBERT bunu nötr görmüş olabilir çünkü başlık...,1,8,43
2,REUTERS_ANN_00003,2006-12-17,"stocks await data, mergers and santa","Hisseler veri, birleşmeler ve Noel Baba rallis...",0,neutral,0.865116,neutral,0.0,0.82,Piyasa bekleyişi anlatılıyor; veri veya birleş...,neutral,same,no,FinBERT etiketi bağlamla uyumlu; 'await' bekle...,1,6,36
3,REUTERS_ANN_00004,2007-01-03,canadian dollar falls to 9-mth lows vs u.s. do...,Kanada doları ABD doları karşısında 9 ayın en ...,-1,negative,0.960498,negative,-1.0,0.98,Para biriminin 9 ayın en düşük seviyesine geri...,negative,same,no,FinBERT etiketi bağlamla uyumlu; 'falls to low...,1,9,50
4,REUTERS_ANN_00005,2007-02-05,"techs buckle under microsoft, wal-mart helps dow",Teknoloji hisseleri Microsoft baskısıyla çöktü...,-1,negative,0.964671,negative,-1.0,0.86,Wal-Mart desteği olumlu olsa da teknoloji hiss...,negative,same,no,FinBERT etiketi bağlamla uyumlu; tech weakness...,1,7,48
5,REUTERS_ANN_00006,2007-03-06,jan factory orders fall 5.6 pct,"Ocak fabrika siparişleri yüzde 5,6 düştü",-1,negative,0.974835,negative,-1.0,0.99,Fabrika siparişlerindeki sert düşüş üretim ve ...,negative,same,no,FinBERT etiketi bağlamla uyumlu; 'orders fall ...,1,6,31
6,REUTERS_ANN_00007,2007-04-18,washington mutual commits $2 bln for subprime ...,Washington Mutual subprime yardımı için 2 mily...,-1,negative,0.816072,negative,-1.0,0.82,Subprime yardımı taahhüdü sorunun büyüklüğünü ...,negative,same,no,FinBERT etiketi bağlamla uyumlu; 'subprime hel...,1,8,50
7,REUTERS_ANN_00008,2007-03-05,google says using cash for merger spree unlikely,Google nakdi birleşme satın alma furyası için ...,0,neutral,0.823399,negative,-1.0,0.76,Nakit kullanarak büyük M&A genişlemesine gitme...,negative,different,yes,FinBERT bunu nötr görmüş olabilir çünkü şirket...,1,8,48
8,REUTERS_ANN_00009,2007-04-27,u.s. automakers not jumping into hd radio,ABD otomobil üreticileri HD radyoya atlamıyor,0,neutral,0.546802,neutral,0.0,0.78,"Yeni teknolojiye geçmeme bilgisi var, ancak fi...",neutral,same,no,FinBERT etiketi bağlamla uyumlu; teknoloji ben...,1,7,41
9,REUTERS_ANN_00010,2007-03-05,"ba shares fall on strike worries, weak market",BA hisseleri grev endişeleri ve zayıf piyasa n...,-1,negative,0.973363,negative,-1.0,0.99,"Hisse düşüşü, grev endişeleri ve zayıf piyasa ...",negative,same,no,FinBERT etiketi bağlamla uyumlu; 'shares fall'...,1,8,45


In [6]:
# final_df'den gerekli Gemini bilgilerini al
gemini_df = final_df[[
    "annotation_id",
    "gemini_label"
]].copy()

# Ana Reuters datasetine annotation_id üzerinden ekle
master_df = reuters_df.merge(
    gemini_df,
    on="annotation_id",
    how="left"
)

print("Birleştirme tamamlandı.")
print("Satır sayısı:", len(master_df))
print("Kolonlar:")
print(master_df.columns.tolist())

display(master_df.head(10))

Birleştirme tamamlandı.
Satır sayısı: 5000
Kolonlar:
['annotation_id', 'date', 'text_en', 'text_tr', 'finbert_score', 'finbert_label', 'finbert_confidence', 'chatgpt_label', 'chatgpt_score', 'chatgpt_confidence', 'chatgpt_reason_tr', 'final_label', 'finbert_correctness', 'review_needed', 'row_note', 'batch_no', 'n_words', 'text_len', 'gemini_label']


,annotation_id,date,text_en,text_tr,finbert_score,finbert_label,finbert_confidence,chatgpt_label,chatgpt_score,chatgpt_confidence,chatgpt_reason_tr,final_label,finbert_correctness,review_needed,row_note,batch_no,n_words,text_len,gemini_label
0,REUTERS_ANN_00001,2007-04-12,s.africa watchdog to probe gold fields bid report,Güney Afrika rekabet kurumu Gold Fields teklif...,-1,negative,0.683211,negative,-1.0,0.86,Bir satın alma teklifinin rekabet kurumu taraf...,negative,same,no,FinBERT etiketi bağlamla uyumlu; 'watchdog to ...,1,8,49,neutral
1,REUTERS_ANN_00002,2007-04-26,new barbie girls sashay into view with mp-3,Yeni Barbie Girls MP3 ile sahneye çıkıyor,0,neutral,0.847032,positive,1.0,0.76,Yeni MP3 özellikli ürün lansmanı ürün yeniliği...,positive,different,yes,FinBERT bunu nötr görmüş olabilir çünkü başlık...,1,8,43,neutral
2,REUTERS_ANN_00003,2006-12-17,"stocks await data, mergers and santa","Hisseler veri, birleşmeler ve Noel Baba rallis...",0,neutral,0.865116,neutral,0.0,0.82,Piyasa bekleyişi anlatılıyor; veri veya birleş...,neutral,same,no,FinBERT etiketi bağlamla uyumlu; 'await' bekle...,1,6,36,neutral
3,REUTERS_ANN_00004,2007-01-03,canadian dollar falls to 9-mth lows vs u.s. do...,Kanada doları ABD doları karşısında 9 ayın en ...,-1,negative,0.960498,negative,-1.0,0.98,Para biriminin 9 ayın en düşük seviyesine geri...,negative,same,no,FinBERT etiketi bağlamla uyumlu; 'falls to low...,1,9,50,negative
4,REUTERS_ANN_00005,2007-02-05,"techs buckle under microsoft, wal-mart helps dow",Teknoloji hisseleri Microsoft baskısıyla çöktü...,-1,negative,0.964671,negative,-1.0,0.86,Wal-Mart desteği olumlu olsa da teknoloji hiss...,negative,same,no,FinBERT etiketi bağlamla uyumlu; tech weakness...,1,7,48,neutral
5,REUTERS_ANN_00006,2007-03-06,jan factory orders fall 5.6 pct,"Ocak fabrika siparişleri yüzde 5,6 düştü",-1,negative,0.974835,negative,-1.0,0.99,Fabrika siparişlerindeki sert düşüş üretim ve ...,negative,same,no,FinBERT etiketi bağlamla uyumlu; 'orders fall ...,1,6,31,negative
6,REUTERS_ANN_00007,2007-04-18,washington mutual commits $2 bln for subprime ...,Washington Mutual subprime yardımı için 2 mily...,-1,negative,0.816072,negative,-1.0,0.82,Subprime yardımı taahhüdü sorunun büyüklüğünü ...,negative,same,no,FinBERT etiketi bağlamla uyumlu; 'subprime hel...,1,8,50,neutral
7,REUTERS_ANN_00008,2007-03-05,google says using cash for merger spree unlikely,Google nakdi birleşme satın alma furyası için ...,0,neutral,0.823399,negative,-1.0,0.76,Nakit kullanarak büyük M&A genişlemesine gitme...,negative,different,yes,FinBERT bunu nötr görmüş olabilir çünkü şirket...,1,8,48,neutral
8,REUTERS_ANN_00009,2007-04-27,u.s. automakers not jumping into hd radio,ABD otomobil üreticileri HD radyoya atlamıyor,0,neutral,0.546802,neutral,0.0,0.78,"Yeni teknolojiye geçmeme bilgisi var, ancak fi...",neutral,same,no,FinBERT etiketi bağlamla uyumlu; teknoloji ben...,1,7,41,neutral
9,REUTERS_ANN_00010,2007-03-05,"ba shares fall on strike worries, weak market",BA hisseleri grev endişeleri ve zayıf piyasa n...,-1,negative,0.973363,negative,-1.0,0.99,"Hisse düşüşü, grev endişeleri ve zayıf piyasa ...",negative,same,no,FinBERT etiketi bağlamla uyumlu; 'shares fall'...,1,8,45,negative


In [7]:
gemini_df = (
    final_df[["annotation_id", "gemini_label"]]
    .drop_duplicates(subset="annotation_id", keep="first")
    .copy()
)

master_df = reuters_df.merge(
    gemini_df,
    on="annotation_id",
    how="left"
)

print("Master kayıt sayısı:", len(master_df))

Master kayıt sayısı: 5000


In [9]:
master_df[["text_en","annotation_id", "gemini_label", "chatgpt_label"]]

,text_en,annotation_id,gemini_label,chatgpt_label
0,s.africa watchdog to probe gold fields bid report,REUTERS_ANN_00001,neutral,negative
1,new barbie girls sashay into view with mp-3,REUTERS_ANN_00002,neutral,positive
2,"stocks await data, mergers and santa",REUTERS_ANN_00003,neutral,neutral
3,canadian dollar falls to 9-mth lows vs u.s. do...,REUTERS_ANN_00004,negative,negative
4,"techs buckle under microsoft, wal-mart helps dow",REUTERS_ANN_00005,neutral,negative
...,...,...,...,...
4995,earnings to lift stocks,REUTERS_ANN_04996,positive,positive
4996,new century stock falls,REUTERS_ANN_04997,negative,negative
4997,icahn seeks motorola to return cash to shareho...,REUTERS_ANN_04998,neutral,positive
4998,hilton hotels posts higher 4th-quarter profit,REUTERS_ANN_04999,positive,positive


In [ ]:
same_count = (
    master_df["gemini_label"] == master_df["chatgpt_label"]
).sum()

print("Aynı etiket sayısı:", same_count)

Aynı etiket sayısı: 3596


In [ ]:
comparison = pd.crosstab(
    master_df["gemini_label"],
    master_df["chatgpt_label"]
)

print(comparison)

chatgpt_label  negative  neutral  positive
gemini_label                              
mixed                 0        2         0
negative           1218      123       162
neutral             255      819       554
positive            106      202      1559


In [ ]:
from sklearn.metrics import cohen_kappa_score

valid = master_df[
    master_df["gemini_label"].notna() &
    master_df["chatgpt_label"].notna()
]

kappa = cohen_kappa_score(
    valid["gemini_label"],
    valid["chatgpt_label"]
)

print("Cohen's Kappa:", kappa)

Cohen's Kappa: 0.5748971305556312
